# 03 - Star Schema Design

In this notebook, we will design the analytical model for the Customer Retention Command Center.

A star schema separates descriptive fields into **dimension tables** and measurable business events into **fact tables**.

Simple way to remember it:

- Dimensions answer: **who, what, where, how, which type?**
- Facts answer: **how many, how much, did it happen, what value?**

For this project, the main fact table will be one customer snapshot. That means each row describes one customer at the time the dataset was captured.

## 1. Import Libraries and Load Cleaned Data

We will use the cleaned CSV exported from notebook 02.

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
clean_file = Path("../data/processed/telco_customer_churn_clean.csv")

df = pd.read_csv(clean_file)

df.head()

,customerid,count,country,state,city,zip_code,lat_long,latitude,longitude,gender,...,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason,is_churned,is_month_to_month,revenue_at_risk
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,53.85,108.15,Yes,1,86,3239,Competitor made better offer,True,True,53.85
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,70.70,151.65,Yes,1,67,2701,Moved,True,True,70.70
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,99.65,820.50,Yes,1,86,5372,Moved,True,True,99.65
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,104.80,3046.05,Yes,1,84,5003,Moved,True,True,104.80
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,103.70,5036.30,Yes,1,89,5340,Competitor had better devices,True,True,103.70


## 2. Review All Columns

Before building tables, we need to see the available fields again.

In [3]:
df.columns.tolist()

['customerid',
 'count',
 'country',
 'state',
 'city',
 'zip_code',
 'lat_long',
 'latitude',
 'longitude',
 'gender',
 'senior_citizen',
 'partner',
 'dependents',
 'tenure_months',
 'phone_service',
 'multiple_lines',
 'internet_service',
 'online_security',
 'online_backup',
 'device_protection',
 'tech_support',
 'streaming_tv',
 'streaming_movies',
 'contract',
 'paperless_billing',
 'payment_method',
 'monthly_charges',
 'total_charges',
 'churn_label',
 'churn_value',
 'churn_score',
 'cltv',
 'churn_reason',
 'is_churned',
 'is_month_to_month',
 'revenue_at_risk']

## 3. Dimension vs Fact Thinking

A **dimension** is usually descriptive. It helps slice, filter, or group the data.

Examples:

- contract type
- payment method
- internet service
- gender
- city

A **fact** usually contains numeric measures or event flags.

Examples:

- monthly charges
- total charges
- churn value
- churn score
- revenue at risk

Some fields can feel ambiguous. That is normal. Data modelling is partly business judgement.

## 4. Your First Classification

Before looking at the suggested model below, try grouping columns yourself.

Write your first attempt in this markdown cell.

**dim_customer:**

- 

**dim_contract:**

- 

**dim_service:**

- 

**dim_payment_method:**

- 

**fact_customer_snapshot:**

- 

## 5. Suggested Column Groups

Here is a first version of the model.

This is not the only possible answer, but it is a practical structure for Power BI.

In [5]:
dim_customer_columns = [
    "customerid",
    "country",
    "state",
    "city",
    "zip_code",
    "lat_long",
    "latitude",
    "longitude",
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
]

dim_contract_columns = [
    "contract",
    "paperless_billing",
]

dim_service_columns = [
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
]

dim_payment_method_columns = [
    "payment_method",
]

fact_customer_snapshot_columns = [
    "customerid",
    "count",
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "churn_label",
    "churn_value",
    "churn_score",
    "cltv",
    "churn_reason",
    "is_churned",
    "is_month_to_month",
    "revenue_at_risk",
]

## 6. Check Whether We Missed Any Columns

This is a useful modelling habit.

Whenever we manually group columns, we should check whether any columns were accidentally left out.

In [6]:
grouped_columns = set(
    dim_customer_columns
    + dim_contract_columns
    + dim_service_columns
    + dim_payment_method_columns
    + fact_customer_snapshot_columns
)

all_columns = set(df.columns)

missing_from_model = sorted(all_columns - grouped_columns)
unknown_to_dataset = sorted(grouped_columns - all_columns)

missing_from_model, unknown_to_dataset

([], [])

**Your notes:**

- Are any columns missing from the model?
- Did we accidentally include any columns that do not exist?
- If something appears, decide whether it belongs in a dimension or fact table.

## 7. Why We Need Keys

Power BI relationships work best when fact tables connect to dimension tables using keys.

We already have `customerid`, but contract, service, and payment method are text fields.

So later we will create keys like:

- `contract_key`
- `service_key`
- `payment_method_key`

The fact table will store these keys instead of repeating long descriptive text everywhere.

## 8. Preview Unique Contract Values

A dimension table should usually contain unique combinations of descriptive fields.

In [7]:
df[dim_contract_columns].drop_duplicates().sort_values(dim_contract_columns).reset_index(drop=True)

,contract,paperless_billing
0,Month-to-month,No
1,Month-to-month,Yes
2,One year,No
3,One year,Yes
4,Two year,No
5,Two year,Yes


**Your notes:**

- How many contract combinations are there?
- Does `paperless_billing` belong with contract, or could it belong somewhere else?
- There is no single perfect answer here. Explain your reasoning.

## 9. Preview Unique Payment Methods

Payment method is a good small dimension because it has a few repeated categories.

In [8]:
df[dim_payment_method_columns].drop_duplicates().sort_values(dim_payment_method_columns).reset_index(drop=True)

,payment_method
0,Bank transfer (automatic)
1,Credit card (automatic)
2,Electronic check
3,Mailed check


## 10. Preview Service Combinations

The service dimension is wider because customers can have many combinations of phone, internet, security, backup, and streaming services.

In [9]:
service_combinations = df[dim_service_columns].drop_duplicates().reset_index(drop=True)

service_combinations.shape

(322, 9)

In [10]:
service_combinations.head(10)

,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies
0,Yes,No,DSL,Yes,Yes,No,No,No,No
1,Yes,No,Fiber optic,No,No,No,No,No,No
2,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes
3,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes
4,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes
5,Yes,No,DSL,No,No,Yes,Yes,No,No
6,No,No phone service,DSL,No,No,Yes,No,No,Yes
7,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service
8,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes
9,No,No phone service,DSL,No,Yes,No,No,No,No


**Your notes:**

- Why are there more service combinations than payment methods?
- Why might this still be useful as a dimension?

## 11. Preview Fact Fields

The fact table will hold the customer-level metrics and flags.

For now, it still includes text fields like `churn_label` and `churn_reason`. Later we can decide whether to move churn reason into its own small dimension.

In [11]:
df[fact_customer_snapshot_columns].head()

,customerid,count,tenure_months,monthly_charges,total_charges,churn_label,churn_value,churn_score,cltv,churn_reason,is_churned,is_month_to_month,revenue_at_risk
0,3668-QPYBK,1,2,53.85,108.15,Yes,1,86,3239,Competitor made better offer,True,True,53.85
1,9237-HQITU,1,2,70.70,151.65,Yes,1,67,2701,Moved,True,True,70.70
2,9305-CDSKC,1,8,99.65,820.50,Yes,1,86,5372,Moved,True,True,99.65
3,7892-POOKP,1,28,104.80,3046.05,Yes,1,84,5003,Moved,True,True,104.80
4,0280-XJGEX,1,49,103.70,5036.30,Yes,1,89,5340,Competitor had better devices,True,True,103.70


## 12. Mini Reflection

Answer these before moving on:

- What is one benefit of separating dimensions and facts?
- Which dimension seems easiest to understand?
- Which dimension seems messiest or most debatable?
- Which fact fields will become important dashboard measures?

## 13. Next Step

In the next notebook, we will physically create these tables:

- `dim_customer.csv`
- `dim_contract.csv`
- `dim_service.csv`
- `dim_payment_method.csv`
- `fact_customer_snapshot.csv`

Then we will load those into SQL and Power BI.